In [ ]:
import logging
from pyspark.sql.functions import col, when, lit, expr

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

try:
    # Step 1: Load Source Data
    logger.info("Loading source data from Unity Catalog table.")
    source_df = spark.table("catalog.source_db.WRK_BIRP_NISS_APRM_DETL")

    # Step 2: Apply Source Qualifier Transformation (SQL Override)
    logger.info("Applying SQL override for initial filtering and transformations.")
    sq_df = source_df.filter(
        (col("ST_ABBR").isin("NY", "NJ") == False) & 
        (col("SOURCE_IND_DERIVED") == "FARMERS")
    ).select(
        col("NISS_APRM_DETL_SK"),
        col("ST_ABBR"),
        col("ST_CD"),
        col("RATNG_CMPY_CD"),
        col("MLT_CAR_IND"),
        col("RT_CLS"),
        col("FINAL_RDRVR_AGE"),
        col("GENDR"),
        col("MRTL_STAT"),
        col("AUTO_USE_CD"),
        col("MILES_TO_WRK"),
        col("GOOD_STDNT_IND"),
        col("DRVR_TRNG_IND"),
        col("SOI_TYP"),
        col("ACCTNG_LOB"),
        col("CVG_TYP_CD"),
        col("REC_EXCPN_IND"),
        col("REC_EXCPN_RSN_DESC")
    )

    # Step 3: Expression Transformation
    logger.info("Applying expression transformations to derive NISS classification codes.")
    exp_df = sq_df.withColumn(
        "NISS_CLASS_CD_FL",
        when(
            (col("ST_ABBR") == "FL") & (col("FINAL_RDRVR_AGE").isNotNull()),
            when(
                (col("FINAL_RDRVR_AGE").cast("int") < 25) & 
                (col("RATNG_CMPY_CD") == "K") & 
                (col("MLT_CAR_IND") == "N") & 
                (col("RT_CLS").rlike("5|5M|9|5F|2F|2M")) & 
                (col("MRTL_STAT") == "M") & 
                (col("AUTO_USE_CD") != "FRM"),
                "1620"
            ).when(
                (col("FINAL_RDRVR_AGE").cast("int") < 25) & 
                (col("RATNG_CMPY_CD") == "K") & 
                (col("MLT_CAR_IND") == "N") & 
                (col("RT_CLS").rlike("5|5M|9|5F|2F|2M")) & 
                (col("MRTL_STAT") == "M") & 
                (col("AUTO_USE_CD") == "FRM"),
                "1623"
            ).otherwise("")
        ).otherwise("")
    )

    # Additional transformations for other derived fields
    exp_df = exp_df.withColumn("AGE", col("FINAL_RDRVR_AGE").cast("int")) \
                   .withColumn("MILES_TO_WRK", col("MILES_TO_WRK").cast("int")) \
                   .withColumn("AUTO_USE_CD", when(col("AUTO_USE_CD").isNull(), "").otherwise(col("AUTO_USE_CD")))

    # Step 4: Update Strategy
    logger.info("Applying update strategy to update NISS_CLASS_CD.")
    upd_df = exp_df.withColumn("NISS_CLASS_CD", col("NISS_CLASS_CD_FL"))

    # Step 5: Write to Target Table
    logger.info("Writing transformed data to Unity Catalog target table.")
    spark.sql("DROP TABLE IF EXISTS catalog.target_db.WRK_BIRP_NISS_APRM_DETL1")
    upd_df.write.format("delta").mode("overwrite").saveAsTable("catalog.target_db.WRK_BIRP_NISS_APRM_DETL1")

    logger.info("ETL workflow completed successfully.")

except Exception as e:
    logger.error(f"Error occurred during ETL workflow: {str(e)}")
    raise
